# 🎂🌍 Birthquake — Interactive App

Find the biggest earthquake(s) that happened on your birthday, using the USGS earthquake catalog via ObsPy, and see the top 5 (by magnitude) plotted on a map, centered on the largest one.

**Why this version, and not the Panel one:** Panel's `pn.ipywidget()` bridge (via `jupyter_bokeh`) is what let a Panel app open automatically in a Sidecar tab, but that bridge is fragile across Bokeh/Panel version combinations — it can throw a `Bokeh.require` JS error that isn't fixable from notebook code. So this version builds the UI directly in `ipywidgets` instead: no custom CSS (just `ipywidgets.Layout`'s built-in `border`/`padding`/`width` properties, which are a structured Python API, not injected stylesheets), and no fragile cross-library JS bridge.

**Opens automatically in a JupyterLab tab — no click required.** The last cell hands the app straight to [`sidecar`](https://github.com/jupyter-widgets/jupyterlab-sidecar) with `anchor="tab-after"`. Because `ipywidgets` and `sidecar` communicate over the same, well-supported kernel comm channel, the tab opens the moment the cell finishes running.

**How to play:** Run all cells in order. The last cell opens the tab automatically.

**If you see "Error displaying widget: model not found":** this means the browser has a display output pointing at a widget that no longer exists in the current kernel — usually because the kernel was restarted, or the notebook was reopened with old saved outputs still attached, without re-running the cells that created the widgets. Fix: **Kernel → Restart Kernel and Run All Cells** (from the very top, including the diagnostic cell). The launch cell also closes any previous Sidecar panel automatically before opening a new one, so re-running just that cell after an edit won't cause this.


In [5]:
# --- Environment check: run this first ---
def _check():
    missing = []
    for pkg in ["obspy", "folium", "pandas", "ipywidgets", "sidecar"]:
        try:
            __import__(pkg)
        except ImportError:
            missing.append(pkg)

    if missing:
        print("⚠️  Missing packages:", ", ".join(missing))
        print(f"    Run: pip install {' '.join(missing)}")
        if "sidecar" in missing:
            print("    'sidecar' opens the app tab automatically. After installing, restart the JupyterLab server (not just the kernel).")
    else:
        print("✅ obspy, folium, pandas, ipywidgets, sidecar are all installed.")
        import ipywidgets, jupyterlab_widgets
        print(f"ipywidgets: {ipywidgets.__version__}  |  jupyterlab_widgets: {jupyterlab_widgets.__version__}")

_check()


✅ obspy, folium, pandas, ipywidgets, sidecar are all installed.
ipywidgets: 8.1.8  |  jupyterlab_widgets: 3.0.16


In [ ]:
import datetime
import io

import pandas as pd
import folium
from obspy import UTCDateTime
from obspy.clients.fdsn import Client
from obspy.clients.fdsn.header import FDSNNoDataException
import ipywidgets as widgets
from IPython.display import display, Image

import matplotlib
matplotlib.use("Agg")  # headless rendering; we display via PNG bytes, not GUI
import matplotlib.pyplot as plt


def get_events(birthday_date):
    """Query the USGS earthquake catalog for the 24 hours starting on birthday_date.

    birthday_date: a datetime.date (e.g. from an ipywidgets.DatePicker)
    Returns a DataFrame sorted by magnitude, descending. Empty if no events found.
    """
    bday_utc = UTCDateTime(birthday_date.year, birthday_date.month, birthday_date.day)

    client = Client("USGS")
    catalog = client.get_events(
        starttime=bday_utc,
        endtime=bday_utc + datetime.timedelta(days=1),
    )

    df = pd.DataFrame([{
        "time": event.origins[0].time,
        "latitude": event.origins[0].latitude,
        "longitude": event.origins[0].longitude,
        "depth": event.origins[0].depth,
        "magnitude": event.magnitudes[0].mag if event.magnitudes else None,
    } for event in catalog])

    if df.empty:
        return df

    df = df.sort_values(by=["magnitude"], ascending=False, na_position="last").reset_index(drop=True)
    return df


def build_map(df):
    """Build a folium map centered on the largest-magnitude event in df.

    Assumes df is already sorted by magnitude, descending (get_events does this),
    so df.iloc[0] is the biggest earthquake and becomes the map's center/marker.
    The map stays centered there rather than re-fitting to the other markers'
    bounds, which would otherwise shift the visual center away from the birthquake.
    """
    top = df.iloc[0]

    m = folium.Map(
        location=[top["latitude"], top["longitude"]],
        zoom_start=5,
        tiles="https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}",
        name="Google Satellite",
        attr="Google",
        overlay=True,
        control=True,
    )

    for i, row in df.iterrows():
        loc = [row["latitude"], row["longitude"]]
        mag = row["magnitude"]
        mag_str = f"{mag:.1f}" if pd.notna(mag) else "unknown"
        popup_html = f"<b>Magnitude {mag_str}</b><br>{row['time']}"

        if i == 0:
            folium.Marker(
                location=loc,
                popup=f"<b>🎂 It's your birthquake!</b><br>{popup_html}",
                tooltip="Your birthquake — click for info",
                icon=folium.Icon(color="green", icon="star"),
            ).add_to(m)
        else:
            folium.CircleMarker(
                location=loc,
                radius=3 + (mag if pd.notna(mag) else 0),
                popup=popup_html,
                color="#ff6600",
                fill=True,
                fill_opacity=0.6,
            ).add_to(m)

    return m


def top_events_table_html(df, n=5):
    """Plain HTML table (inline-styled borders, no external stylesheet) of
    the top n events by magnitude."""
    cell_style = "border: 1px solid #888; padding: 6px 10px;"
    rows_html = ""
    for pos, (_, row) in enumerate(df.head(n).iterrows(), start=1):
        mag = row["magnitude"]
        mag_str = f"{mag:.1f}" if pd.notna(mag) else "—"
        depth_km = f"{row['depth'] / 1000:.1f} km" if pd.notna(row["depth"]) else "—"
        marker = "🎂" if pos == 1 else str(pos)
        rows_html += (
            f"<tr><td style='{cell_style}'>{marker}</td>"
            f"<td style='{cell_style}'>{row['time']}</td>"
            f"<td style='{cell_style}'>{row['latitude']:.2f}, {row['longitude']:.2f}</td>"
            f"<td style='{cell_style}'>{depth_km}</td>"
            f"<td style='{cell_style}'>{mag_str}</td></tr>"
        )
    header_cells = "".join(
        f"<th style='{cell_style}'>{label}</th>"
        for label in ["#", "Time (UTC)", "Location", "Depth", "Magnitude"]
    )
    return (
        "<table style='border-collapse: collapse;'>"
        f"<tr>{header_cells}</tr>"
        f"{rows_html}"
        "</table>"
    )


# Fallback chain of long-running, well-maintained Global Seismographic Network
# stations (network, station, location, channel). Any single station can have
# gaps -- maintenance outages, instrument swaps, old dates before it existed --
# so we try each in turn and use the first one with data for the requested
# window, rather than failing just because the first choice has a gap.
WAVEFORM_STATION_FALLBACKS = [
    ("IU", "ANMO", "00", "LHZ"),  # Albuquerque, NM, USA
    ("IU", "HRV", "00", "LHZ"),   # Harvard, MA, USA
    ("II", "PFO", "00", "LHZ"),   # Pinon Flat, CA, USA
    ("IU", "COLA", "00", "LHZ"),  # College, AK, USA
    ("IU", "KONO", "00", "LHZ"),  # Kongsberg, Norway
]


def get_birthquake_waveforms(
    df,
    network=None,
    station=None,
    location=None,
    channel=None,
    seconds_before=120,
    seconds_after=600,
    save_path=None,
):
    """Fetch MiniSEED waveform data around the birthquake's origin time.

    "The birthquake" is df.iloc[0] -- the largest-magnitude event -- since
    get_events() already sorts by magnitude, descending.

    Queries the EarthScope FDSN dataselect service (the modern replacement
    for the deprecated service.iris.edu endpoints). If network/station/
    location/channel are all left as None, tries each station in
    WAVEFORM_STATION_FALLBACKS in turn and returns data from the first one
    that has coverage for the requested window -- any single global station
    can have a data gap for a given birthquake's time, so this avoids failing
    outright just because the first pick (IU.ANMO) happens to be down.
    Pass explicit network/station/location/channel to fetch one specific
    station instead (e.g. a station nearer the epicenter).

    Returns an obspy Stream, and writes it to a local .mseed file
    (auto-named from the station and origin time unless save_path is given).
    """
    top = df.iloc[0]
    origin_time = UTCDateTime(str(top["time"]))
    starttime = origin_time - seconds_before
    endtime = origin_time + seconds_after

    if any(v is not None for v in (network, station, location, channel)):
        candidates = [(network, station, location, channel)]
    else:
        candidates = WAVEFORM_STATION_FALLBACKS

    waveform_client = Client(base_url="http://service.earthscope.org")

    st = None
    last_exc = None
    tried = []
    for net, sta, loc, chan in candidates:
        tried.append(f"{net}.{sta}.{loc}.{chan}")
        try:
            st = waveform_client.get_waveforms(
                network=net,
                station=sta,
                location=loc,
                channel=chan,
                starttime=starttime,
                endtime=endtime,
            )
            network, station, location, channel = net, sta, loc, chan
            break
        except FDSNNoDataException as exc:
            last_exc = exc
            continue

    if st is None:
        raise FDSNNoDataException(
            f"No data available for request at any of: {', '.join(tried)}"
        ) from last_exc

    if save_path is None:
        date_tag = origin_time.strftime("%Y%m%dT%H%M%S")
        save_path = f"birthquake_{network}_{station}_{channel}_{date_tag}.mseed"

    st.write(save_path, format="MSEED")
    print(f"Saved {len(st)} trace(s) to {save_path} (from {network}.{station}.{location}.{channel})")
    return st


print("Core functions loaded.")


In [7]:
class BirthquakeApp:
    """A small, self-contained ipywidgets 'app'. Uses only ipywidgets.Layout's
    built-in border/padding/width properties for structure -- no injected CSS,
    no Bokeh/jupyter_bokeh bridge."""

    CARD_LAYOUT = dict(
        border="1px solid #d0d7de",
        padding="16px",
        width="700px",
    )

    def __init__(self):
        self.last_date = None
        self.root = widgets.VBox(layout=widgets.Layout(width="100%", align_items="center"))
        self.show_form()

    # ---------- public entry point ----------
    def display(self):
        display(self.root)

    # ---------- screen: form ----------
    def show_form(self, *_):
        title = widgets.HTML("<h2>\U0001F382 Birthquake</h2><p>Find the biggest earthquake on your birthday</p>")

        self.date_picker = widgets.DatePicker(
            description="Birthday",
            value=self.last_date,
            max=datetime.date.today() - datetime.timedelta(days=1),
        )
        self.find_btn = widgets.Button(description="Find My Birthquake", button_style="success", icon="search")
        self.find_btn.on_click(self.on_find_clicked)

        input_row = widgets.HBox(
            [self.date_picker, self.find_btn],
            layout=widgets.Layout(justify_content="center", align_items="center", margin="8px 0 0 0"),
        )
        self.status = widgets.HTML("")

        card = widgets.VBox([title, input_row, self.status], layout=widgets.Layout(**self.CARD_LAYOUT))
        self.root.children = [card]

    def on_find_clicked(self, _btn):
        if self.date_picker.value is None:
            self.status.value = "<p>\u26a0\ufe0f Please pick a date first.</p>"
            return

        if self.date_picker.value >= datetime.date.today():
            self.status.value = "<p>\u26a0\ufe0f Please pick a date before today.</p>"
            return

        self.last_date = self.date_picker.value
        self.find_btn.disabled = True
        self.status.value = "<p>\U0001F50E Searching the USGS earthquake catalog\u2026</p>"

        try:
            df = get_events(self.last_date)
        except Exception as exc:
            self.find_btn.disabled = False
            self.status.value = f"<p>\u274c Something went wrong querying USGS: {exc}</p>"
            return

        self.find_btn.disabled = False

        if df.empty:
            self.status.value = "<p>\u26a0\ufe0f No earthquakes found for that day \u2014 try another date.</p>"
            return

        self.status.value = ""
        self.show_results(df)

    # ---------- screen: results ----------
    def show_results(self, df):
        top5 = df.head(5).reset_index(drop=True)
        self.current_df = top5  # so the waveform button knows which birthquake to fetch
        date_str = self.last_date.strftime("%B %d, %Y")

        title = widgets.HTML(
            "<div style='text-align:center'>"
            f"<h2>\U0001F30D Your Birthquake</h2><p>Earthquakes on {date_str}</p>"
            "<p><b>Top 5 earthquakes that day, by magnitude:</b></p>"
            "</div>"
        )
        table = widgets.HTML(
            f"<div style='display:flex; justify-content:center'>{top_events_table_html(top5, n=5)}</div>"
        )

        map_output = widgets.Output(layout=widgets.Layout(border="1px solid #d0d7de"))
        with map_output:
            display(build_map(top5))

        again_btn = widgets.Button(description="Search Another Date", button_style="info", icon="redo")
        again_btn.on_click(self.show_form)

        self.waveform_btn = widgets.Button(description="View Waveform", button_style="warning", icon="wave-square")
        self.waveform_btn.on_click(self.on_view_waveform_clicked)

        buttons = widgets.HBox(
            [again_btn, self.waveform_btn],
            layout=widgets.Layout(justify_content="center", margin="10px 0 0 0"),
        )

        self.waveform_status = widgets.HTML("")
        self.waveform_output = widgets.Output(layout=widgets.Layout(border="1px solid #d0d7de", margin="10px 0 0 0"))

        card = widgets.VBox(
            [title, table, map_output, buttons, self.waveform_status, self.waveform_output],
            layout=widgets.Layout(**self.CARD_LAYOUT),
        )
        self.root.children = [card]

    def on_view_waveform_clicked(self, _btn):
        self.waveform_btn.disabled = True
        self.waveform_status.value = "<p>\U0001F50E Fetching waveform data\u2026</p>"
        self.waveform_output.clear_output()

        try:
            st = get_birthquake_waveforms(self.current_df)
        except Exception as exc:
            self.waveform_btn.disabled = False
            self.waveform_status.value = f"<p>\u274c Could not fetch waveform data: {exc}</p>"
            return

        fig = st.plot(handle=True, show=False)
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=100, bbox_inches="tight")
        buf.seek(0)
        plt.close(fig)

        self.waveform_btn.disabled = False
        self.waveform_status.value = ""
        with self.waveform_output:
            display(Image(data=buf.getvalue()))


In [8]:
from sidecar import Sidecar

# Close any previously-opened Sidecar panel before making a new one. Without
# this, re-running this cell (e.g. after editing BirthquakeApp above) leaves
# the old panel's widget orphaned in the frontend -- which is one of the most
# common causes of "Error displaying widget: model not found".
try:
    sidecar.close()
except NameError:
    pass
except Exception:
    pass

app = BirthquakeApp()

sidecar = Sidecar(title="Birthquake", anchor="tab-after")
with sidecar:
    app.display()

print("Birthquake opened automatically in its own JupyterLab tab.")


Birthquake opened automatically in its own JupyterLab tab.
